# LeetCode 1913 — Maximum Product Difference Between Two Pairs bằng Python

Link đề: https://leetcode.com/problems/maximum-product-difference-between-two-pairs/

Độ khó: **Easy** · Chủ đề: Array, Sorting

```python
class Solution:
    def maxProductDifference(self, nums: List[int]) -> int:

```

**Mục tiêu học:**

1. Đọc hiểu "chọn 4 chỉ số **phân biệt**" và biểu thức `(a*b) - (c*d)`.
2. Brute force thử mọi bộ 4 chỉ số `O(n⁴)` — dùng làm "oracle" (đáp án chuẩn) để kiểm thử.
3. Insight: với số **dương**, muốn hiệu lớn nhất thì `a*b` lớn nhất (2 số lớn nhất) và `c*d` nhỏ nhất (2 số nhỏ nhất).
4. Hai cách tối ưu: **sắp xếp** `O(n log n)` và **một lượt duyệt theo dõi top-2 / bottom-2** `O(n)`; thêm `heapq`.
5. Hiểu vì sao điều kiện `nums[i] >= 1` quan trọng — với số âm thì insight sụp đổ.

## 1. Bài toán yêu cầu gì?

**Product difference** của hai cặp `(a, b)` và `(c, d)` là `(a * b) - (c * d)`.

Cho mảng `nums`, chọn **4 chỉ số khác nhau** `w, x, y, z` để `(nums[w]*nums[x]) - (nums[y]*nums[z])` **lớn nhất**. Trả về giá trị lớn nhất đó.

| nums | Kết quả | Cách chọn |
|---|---:|---|
| `[5,6,2,7,4]` | 34 | `(6*7) - (2*4) = 42 - 8` |
| `[4,2,5,9,7,4,8]` | 64 | `(9*8) - (2*4) = 72 - 8` |
| `[1,1,1,1]` | 0 | `1*1 - 1*1` |

**Ràng buộc:** `4 <= len(nums) <= 10^4`, `1 <= nums[i] <= 10^4` — mọi số đều **dương**.

Trên LeetCode, chữ ký dùng `List[int]` nên cần `from typing import List` khi chạy ngoài LeetCode.

In [1]:
from typing import List

examples = [
    ([5, 6, 2, 7, 4], 34),
    ([4, 2, 5, 9, 7, 4, 8], 64),
    ([1, 1, 1, 1], 0),
]
for nums, expected in examples:
    print(f"{nums} -> kỳ vọng {expected}")

[5, 6, 2, 7, 4] -> kỳ vọng 34
[4, 2, 5, 9, 7, 4, 8] -> kỳ vọng 64
[1, 1, 1, 1] -> kỳ vọng 0


## 2. Nền tảng Python cần biết

### 2.1. `from typing import List` — gợi ý kiểu (type hint)

`List[int]` chỉ là **ghi chú** "đây là list các số nguyên"; Python không kiểm tra lúc chạy. Từ Python 3.9 có thể viết `list[int]` mà không cần import; LeetCode vẫn dùng kiểu cũ `List`.

In [2]:
def demo(xs: List[int]) -> int:
    return sum(xs)

print(demo([1, 2, 3]))
print(demo.__annotations__)

6
{'xs': typing.List[int], 'return': <class 'int'>}


### 2.2. `sorted()` và `list.sort()`

- `sorted(nums)` trả về list **mới** đã sắp tăng dần; `nums` không đổi.
- `nums.sort()` sắp **tại chỗ**, trả về `None`.
- Chỉ số âm: `a[-1]` là phần tử cuối, `a[-2]` là áp chót.

In [3]:
nums = [5, 6, 2, 7, 4]
s = sorted(nums)
print("sorted:", s, "| nums gốc:", nums)
print("2 nhỏ nhất:", s[0], s[1], "| 2 lớn nhất:", s[-1], s[-2])

r = nums.sort()
print("nums.sort() trả về", r, "và nums giờ là", nums)

sorted: [2, 4, 5, 6, 7] | nums gốc: [5, 6, 2, 7, 4]
2 nhỏ nhất: 2 4 | 2 lớn nhất: 7 6
nums.sort() trả về None và nums giờ là [2, 4, 5, 6, 7]


### 2.3. `float('inf')` — giá trị "vô cực" để khởi tạo min/max

Khi tìm số nhỏ nhất, khởi tạo `small = float('inf')` để số đầu tiên chắc chắn nhỏ hơn. Tìm lớn nhất thì khởi tạo `0` (vì đề cho `nums[i] >= 1`) hoặc `float('-inf')`.

In [4]:
inf = float("inf")
print(3 < inf, -10**18 > float("-inf"))
print(min(inf, 7), max(float("-inf"), 7))

True True
7 7


### 2.4. `heapq.nlargest` / `nsmallest`

`heapq.nlargest(k, nums)` trả về `k` phần tử lớn nhất (giảm dần), `heapq.nsmallest(k, nums)` trả về `k` phần tử nhỏ nhất (tăng dần) — không cần sắp toàn bộ.

In [5]:
import heapq

nums = [4, 2, 5, 9, 7, 4, 8]
print(heapq.nlargest(2, nums), heapq.nsmallest(2, nums))

[9, 8] [2, 4]


### 2.5. `itertools.permutations` — liệt kê bộ chỉ số phân biệt

`permutations(range(n), 4)` sinh mọi bộ 4 chỉ số **khác nhau có thứ tự**. Dùng cho brute force.

In [6]:
from itertools import permutations

print(list(permutations(range(4), 2))[:6], "...")
print("số bộ 4 chỉ số khi n=5:", len(list(permutations(range(5), 4))))   # 5*4*3*2

[(0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3)] ...
số bộ 4 chỉ số khi n=5: 120


## 3. Tính bằng tay — ta muốn gì?

Hiệu `A - B` lớn nhất khi `A` **lớn nhất** và `B` **nhỏ nhất**.

- `A = a * b`: với số dương, tích lớn nhất của 2 số là tích **2 số lớn nhất**.
- `B = c * d`: tích nhỏ nhất là tích **2 số nhỏ nhất**.

`[5,6,2,7,4]` sắp xếp → `[2,4,5,6,7]`:

- 2 lớn nhất: `7, 6` → `42`.
- 2 nhỏ nhất: `2, 4` → `8`.
- `42 - 8 = 34` ✓.

Bốn số này lấy từ **4 vị trí khác nhau** trong mảng đã sắp (vị trí 0, 1, n-2, n-1) — vì `n >= 4` nên không trùng.

In [7]:
nums = [5, 6, 2, 7, 4]
s = sorted(nums)
A = s[-1] * s[-2]
B_ = s[0] * s[1]
print(f"A = {s[-1]}*{s[-2]} = {A}, B = {s[0]}*{s[1]} = {B_}, A - B = {A - B_}")

A = 7*6 = 42, B = 2*4 = 8, A - B = 34


## 4. Cách 1 — Brute force: thử mọi bộ 4 chỉ số

Liệt kê mọi `(w, x, y, z)` phân biệt, tính hiệu, lấy max. Không cần insight gì — dùng làm **oracle** kiểm thử.

In [8]:
def max_product_diff_brute(nums):
    best = float("-inf")
    for w, x, y, z in permutations(range(len(nums)), 4):
        val = nums[w] * nums[x] - nums[y] * nums[z]
        if val > best:
            best = val
    return best

for nums, expected in examples:
    print(nums, "->", max_product_diff_brute(nums), "| kỳ vọng", expected)

[5, 6, 2, 7, 4] -> 34 | kỳ vọng 34
[4, 2, 5, 9, 7, 4, 8] -> 64 | kỳ vọng 64
[1, 1, 1, 1] -> 0 | kỳ vọng 0


**Độ phức tạp:** `n·(n-1)·(n-2)·(n-3) = O(n⁴)`. Với `n = 10^4` là ~10^16 phép — hoàn toàn không khả thi. Với `n = 7` chỉ 840 bộ — đủ để kiểm thử.

**Nút thắt:** ta thử mọi tổ hợp, trong khi phân tích ở mục 3 cho thấy chỉ có **4 số** quyết định kết quả.

## 5. Cách 2 — Sắp xếp rồi lấy hai đầu

In [9]:
def max_product_diff_sort(nums):
    s = sorted(nums)
    return s[-1] * s[-2] - s[0] * s[1]

for nums, expected in examples:
    print(nums, "->", max_product_diff_sort(nums), "| kỳ vọng", expected)

[5, 6, 2, 7, 4] -> 34 | kỳ vọng 34
[4, 2, 5, 9, 7, 4, 8] -> 64 | kỳ vọng 64
[1, 1, 1, 1] -> 0 | kỳ vọng 0


### Code nộp LeetCode — Cách 2

In [10]:
class Solution:
    def maxProductDifference(self, nums: List[int]) -> int:
        nums.sort()
        return nums[-1] * nums[-2] - nums[0] * nums[1]


print(Solution().maxProductDifference([5, 6, 2, 7, 4]))   # 34

34


**Độ phức tạp:** sắp xếp `O(n log n)`; bộ nhớ `O(n)` nếu dùng `sorted` (hoặc `O(1)` phụ thêm với `nums.sort()`, dù Timsort có thể dùng tới `O(n)` bộ nhớ tạm). Lưu ý `nums.sort()` **sửa** mảng của người gọi — trên LeetCode chấp nhận được.

**Câu hỏi:** ta sắp **toàn bộ** mảng chỉ để lấy 4 phần tử. Có cần không?

## 6. Lời giải chính — Một lượt duyệt, theo dõi 2 lớn nhất và 2 nhỏ nhất

Giữ 4 biến:

- `big1 >= big2`: hai số lớn nhất đã thấy.
- `small1 <= small2`: hai số nhỏ nhất đã thấy.

Với mỗi `x`:

**Cập nhật "lớn":**

```python
if x > big1:      big2 = big1; big1 = x     # x thành số lớn nhất, big1 cũ lùi xuống hạng 2
elif x > big2:    big2 = x                  # x chỉ vượt hạng 2
```

**Cập nhật "nhỏ"** (đối xứng):

```python
if x < small1:    small2 = small1; small1 = x
elif x < small2:  small2 = x
```

Thứ tự gán quan trọng: phải đẩy `big1` cũ xuống `big2` **trước** khi ghi đè `big1`.

In [11]:
def max_product_diff_one_pass(nums):
    big1 = big2 = 0                         # nums[i] >= 1 nên 0 là "nhỏ hơn mọi số"
    small1 = small2 = float("inf")
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        elif x > big2:
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
    return big1 * big2 - small1 * small2

for nums, expected in examples:
    print(nums, "->", max_product_diff_one_pass(nums), "| kỳ vọng", expected)

[5, 6, 2, 7, 4] -> 34 | kỳ vọng 34
[4, 2, 5, 9, 7, 4, 8] -> 64 | kỳ vọng 64
[1, 1, 1, 1] -> 0 | kỳ vọng 0


### Vì sao `if ... elif` chứ không phải hai `if` riêng?

Nếu viết hai `if` độc lập, khi `x > big1`, ta gán `big1 = x` rồi `if x > big2` cũng đúng → `big2 = x` → `big2` bị ghi đè bằng chính `x`, mất số lớn thứ hai thật.

### Vì sao phần "lớn" và phần "nhỏ" dùng hai `if` riêng (không `elif` giữa chúng)?

Một số có thể vừa thuộc top-2 vừa thuộc bottom-2 trong lúc đang duyệt (ví dụ ở những phần tử đầu tiên). Hai nhóm cập nhật **độc lập**. Cuối cùng, vì `n >= 4`, bốn vai trò rơi vào 4 vị trí khác nhau (xem mục 10).

### Trường hợp có giá trị trùng

Với `[4, 4, 1, 1]`: `big1 = 4`, rồi `x = 4` không `> big1` nhưng `> big2 = 0` → `big2 = 4`. Đúng: hai số lớn nhất là `4, 4` (hai vị trí khác nhau).

In [12]:
print(max_product_diff_one_pass([4, 4, 1, 1]), "==", max_product_diff_brute([4, 4, 1, 1]))

15 == 15


## 7. TRACE từng bước

Ví dụ `[4,2,5,9,7,4,8]`:

| x | big1 | big2 | small1 | small2 | Ghi chú |
|---:|---:|---:|---:|---:|---|
| 4 | 4 | 0 | 4 | inf | khởi đầu |
| 2 | 4 | 2 | 2 | 4 | 2 vào hạng 2 "lớn"; 2 thành nhỏ nhất |
| 5 | 5 | 4 | 2 | 4 | 5 lên đầu, 4 lùi xuống |
| 9 | 9 | 5 | 2 | 4 | |
| 7 | 9 | 7 | 2 | 4 | 7 vượt hạng 2 |
| 4 | 9 | 7 | 2 | 4 | 4 không `< small2 = 4` → không đổi |
| 8 | 9 | 8 | 2 | 4 | |

Kết quả: `9*8 - 2*4 = 72 - 8 = 64`.

In [13]:
def trace_one_pass(nums):
    big1 = big2 = 0
    small1 = small2 = float("inf")
    print(f"{'x':>3} | {'big1':>4} {'big2':>4} | {'small1':>6} {'small2':>6}")
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        elif x > big2:
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
        print(f"{x:>3} | {big1:>4} {big2:>4} | {small1:>6} {str(small2):>6}")
    res = big1 * big2 - small1 * small2
    print(f"=> {big1}*{big2} - {small1}*{small2} = {res}")
    return res

trace_one_pass([4, 2, 5, 9, 7, 4, 8])

  x | big1 big2 | small1 small2
  4 |    4    0 |      4    inf
  2 |    4    2 |      2      4
  5 |    5    4 |      2      4
  9 |    9    5 |      2      4
  7 |    9    7 |      2      4
  4 |    9    7 |      2      4
  8 |    9    8 |      2      4
=> 9*8 - 2*4 = 64


64

### Trace ca toàn giá trị bằng nhau và ca giảm dần

In [14]:
trace_one_pass([1, 1, 1, 1])
print()
trace_one_pass([10, 8, 6, 4, 2])

  x | big1 big2 | small1 small2
  1 |    1    0 |      1    inf
  1 |    1    1 |      1      1
  1 |    1    1 |      1      1
  1 |    1    1 |      1      1
=> 1*1 - 1*1 = 0

  x | big1 big2 | small1 small2
 10 |   10    0 |     10    inf
  8 |   10    8 |      8     10
  6 |   10    8 |      6      8
  4 |   10    8 |      4      6
  2 |   10    8 |      2      4
=> 10*8 - 2*4 = 72


72

## 8. Cách 4 — `heapq` (gọn, không sắp toàn bộ)

In [15]:
def max_product_diff_heap(nums):
    a, b_ = heapq.nlargest(2, nums)
    c_, d = heapq.nsmallest(2, nums)
    return a * b_ - c_ * d

for nums, expected in examples:
    print(nums, "->", max_product_diff_heap(nums), "| kỳ vọng", expected)

[5, 6, 2, 7, 4] -> 34 | kỳ vọng 34
[4, 2, 5, 9, 7, 4, 8] -> 64 | kỳ vọng 64
[1, 1, 1, 1] -> 0 | kỳ vọng 0


`heapq.nlargest(k, ...)` chạy `O(n log k)`; với `k = 2` coi như `O(n)`. Viết gọn, nhưng hằng số lớn hơn vòng lặp tay một chút.

## 9. Code nộp LeetCode — lời giải chính (một lượt)

In [16]:
class Solution:
    def maxProductDifference(self, nums: List[int]) -> int:
        big1 = big2 = 0
        small1 = small2 = float("inf")
        for x in nums:
            if x > big1:
                big2, big1 = big1, x
            elif x > big2:
                big2 = x
            if x < small1:
                small2, small1 = small1, x
            elif x < small2:
                small2 = x
        return big1 * big2 - small1 * small2


sol = Solution()
for nums, expected in examples:
    assert sol.maxProductDifference(nums[:]) == expected
print("Solution (một lượt) PASS")

Solution (một lượt) PASS


## 10. Vì sao đúng?

Gọi mảng đã sắp là `s[0] <= s[1] <= ... <= s[n-1]`.

**Cận trên.** Với mọi lựa chọn hợp lệ, `a*b <= s[n-1]*s[n-2]` (tích hai phần tử ở hai vị trí khác nhau không vượt tích hai số lớn nhất, vì mọi số dương). Tương tự `c*d >= s[0]*s[1]`. Vậy `a*b - c*d <= s[n-1]*s[n-2] - s[0]*s[1]`.

**Đạt được.** Chọn 4 vị trí `n-1, n-2, 0, 1` trong mảng đã sắp. Vì `n >= 4`, bốn vị trí này **phân biệt** (`1 < n-2`). Nên giá trị cận trên đạt được.

Tại sao cần **số dương**? Nếu có số âm, tích hai số âm lớn có thể là dương rất lớn, và "tích nhỏ nhất" có thể là (âm lớn) × (dương lớn). Insight "2 lớn nhất / 2 nhỏ nhất" không còn đúng (xem biến thể).

**Một lượt duyệt đúng.** Invariant: sau khi duyệt tiền tố `nums[:k]`, `(big1, big2)` là hai giá trị lớn nhất (tính cả trùng lặp, ở hai vị trí khác nhau) của tiền tố đó; tương tự `(small1, small2)`. Mỗi bước `if/elif` chèn `x` vào đúng hạng. Khi `k = n` ta có đúng `s[n-1], s[n-2], s[0], s[1]`.

## 11. Độ phức tạp và bảng so sánh

`n = len(nums)`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Ghi chú |
|---|---|:---:|:---:|---|
| 1. Brute force | mọi bộ 4 chỉ số | `O(n⁴)` | `O(1)` | oracle kiểm thử |
| 2. Sắp xếp | hai đầu của mảng đã sắp | `O(n log n)` | `O(n)` hoặc tại chỗ | ngắn, dễ nhớ |
| 3. Một lượt top-2/bottom-2 | 4 biến | `O(n)` | `O(1)` | tối ưu |
| 4. `heapq` | `nlargest/nsmallest(2)` | `O(n)` (k=2) | `O(1)` | gọn |

Giá trị lớn nhất có thể: `10^4 * 10^4 = 10^8` — vừa trong `int` 32-bit.

## 12. Bộ test kiểm tra

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:30} PASS")

fast = [max_product_diff_sort, max_product_diff_one_pass, max_product_diff_heap]
tests = [
    ("ví dụ 1", [5, 6, 2, 7, 4], 34),
    ("ví dụ 2", [4, 2, 5, 9, 7, 4, 8], 64),
    ("toàn bằng nhau", [1, 1, 1, 1], 0),
    ("đúng 4 phần tử", [1, 2, 3, 4], 10),
    ("tăng dần", [1, 2, 3, 4, 5, 6], 28),
    ("giảm dần", [10, 8, 6, 4, 2], 72),
    ("trùng ở đỉnh", [4, 4, 1, 1], 15),
    ("lớn nhất ở đầu", [10000, 1, 1, 10000], 10**8 - 1),
]
for name, nums, e in tests:
    for f in fast:
        assert f(nums[:]) == e, (name, f.__name__)
    assert max_product_diff_brute(nums) == e, name
    check(name + " (4 cách)", max_product_diff_one_pass(nums), e)

ví dụ 1 (4 cách)               PASS
ví dụ 2 (4 cách)               PASS
toàn bằng nhau (4 cách)        PASS
đúng 4 phần tử (4 cách)        PASS
tăng dần (4 cách)              PASS
giảm dần (4 cách)              PASS
trùng ở đỉnh (4 cách)          PASS
lớn nhất ở đầu (4 cách)        PASS


In [18]:
import random

random.seed(1913)
mismatch = 0
for _ in range(1500):
    n_ = random.randint(4, 7)
    nums = [random.randint(1, 20) for _ in range(n_)]
    expected = max_product_diff_brute(nums)
    if any(f(nums[:]) != expected for f in fast):
        mismatch += 1
print("Số ca lệch so với brute force:", mismatch)
assert mismatch == 0

Số ca lệch so với brute force: 0


In [19]:
import time

big = [random.randint(1, 10**4) for _ in range(10**4)]
t0 = time.perf_counter(); r1 = max_product_diff_one_pass(big); t1 = time.perf_counter()
r2 = max_product_diff_sort(big)
print("n = 10^4: một lượt =", r1, "| sắp xếp =", r2, "| thời gian một lượt %.4f s" % (t1 - t0))
assert r1 == r2

n = 10^4: một lượt = 99999997 | sắp xếp = 99999997 | thời gian một lượt 0.0010 s


## 13. Lỗi người mới hay gặp

### Lỗi 1: Hai `if` thay vì `if/elif` khi cập nhật top-2

In [20]:
def wrong_two_ifs(nums):
    big1 = big2 = 0
    small1 = small2 = float("inf")
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        if x > big2:            # SAI: chạy luôn sau khi vừa gán big1 = x
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
    return big1 * big2 - small1 * small2

print("Sai :", wrong_two_ifs([5, 6, 2, 7, 4]), "| Đúng:", max_product_diff_one_pass([5, 6, 2, 7, 4]))

Sai : 41 | Đúng: 34


Vì sao sai: với `x = 7`, sau khi `big1 = 7`, điều kiện `7 > big2` cũng đúng nên `big2 = 7` — coi như dùng số 7 **hai lần**.

### Lỗi 2: Ghi đè `big1` trước khi đẩy xuống `big2`

```python
if x > big1:
    big1 = x
    big2 = big1      # SAI: big2 nhận giá trị MỚI, mất số lớn cũ
```

Sửa: `big2, big1 = big1, x` (gán song song) hoặc gán `big2 = big1` **trước**.

In [21]:
def wrong_order(nums):
    big1 = big2 = 0
    small1 = small2 = float("inf")
    for x in nums:
        if x > big1:
            big1 = x
            big2 = big1        # SAI
        elif x > big2:
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
    return big1 * big2 - small1 * small2

print("Sai :", wrong_order([5, 6, 2, 7, 4]), "| Đúng:", max_product_diff_one_pass([5, 6, 2, 7, 4]))

Sai : 41 | Đúng: 34


### Lỗi 3: Khởi tạo `small1 = small2 = 0`

Vì mọi số `>= 1`, không số nào `< 0` → `small1, small2` mãi là 0 → trả về `big1*big2` (sai). Phải khởi tạo bằng `float('inf')` (hoặc một số lớn hơn mọi giá trị, như `10**4 + 1`).

In [22]:
def wrong_init(nums):
    big1 = big2 = 0
    small1 = small2 = 0            # SAI
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        elif x > big2:
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
    return big1 * big2 - small1 * small2

print("Sai :", wrong_init([5, 6, 2, 7, 4]), "| Đúng:", max_product_diff_one_pass([5, 6, 2, 7, 4]))

Sai : 42 | Đúng: 34


### Lỗi 4: Dùng `set` để "lấy 2 lớn nhất" — làm mất số trùng

```python
s = sorted(set(nums))      # SAI: [4, 4, 1, 1] -> [1, 4]
```

Hai vị trí khác nhau có thể cùng giá trị; đề chọn **chỉ số** phân biệt, không phải **giá trị** phân biệt.

In [23]:
nums = [4, 4, 1, 1]
print("sorted(set):", sorted(set(nums)), "-> không đủ 4 số, ý tưởng sai")
print("Đúng:", max_product_diff_sort(nums))

sorted(set): [1, 4] -> không đủ 4 số, ý tưởng sai
Đúng: 15


## 14. Mẹo nhận dạng pattern và biến thể

- "Chỉ cần vài phần tử **cực trị** (lớn nhất/nhỏ nhất)" → không cần sắp toàn bộ; giữ **top-k** bằng vài biến (`k` nhỏ) hoặc `heapq` (`k` lớn hơn).
- Bài có tích + điều kiện **số dương** → cực trị ở hai đầu mảng đã sắp. Khi có số âm → phải xét thêm tổ hợp "âm × âm".

Bài liên quan: 628 (Maximum Product of Three Numbers — có số âm!), 1464 (Maximum Product of Two Elements in an Array), 414 (Third Maximum Number), 215 (Kth Largest Element).

### Biến thể: nếu cho phép số âm?

Khi đó tích lớn nhất có thể là `s[0]*s[1]` (hai số âm), tích nhỏ nhất có thể là `s[0]*s[n-1]`. Cách an toàn: thử mọi cách gán 4 vai trò cho các phần tử **ở hai đầu** mảng đã sắp (đủ để chứa các ứng viên cực trị), đối chiếu brute force.

In [24]:
def max_product_diff_with_negatives(nums):
    s = sorted(nums)
    n = len(s)
    cand_idx = sorted(set(list(range(min(4, n))) + list(range(max(0, n - 4), n))))
    best = float("-inf")
    for w, x, y, z in permutations(cand_idx, 4):
        best = max(best, s[w] * s[x] - s[y] * s[z])
    return best

random.seed(7)
for _ in range(1000):
    arr = [random.randint(-10, 10) for _ in range(random.randint(4, 7))]
    assert max_product_diff_with_negatives(arr) == max_product_diff_brute(arr), arr
print("Ví dụ có số âm:", [-10, -9, 1, 2, 3], "->", max_product_diff_with_negatives([-10, -9, 1, 2, 3]),
      "| công thức cũ cho", max_product_diff_sort([-10, -9, 1, 2, 3]))

Ví dụ có số âm: [-10, -9, 1, 2, 3] -> 88 | công thức cũ cho -84


Với `[-10,-9,1,2,3]`: công thức cũ cho `3*2 - (-10)*(-9) = 6 - 90 = -84`, trong khi đáp án đúng là `(-10)*(-9) - 1*2 = 88` (hai số âm cho tích dương lớn nhất). Bài học: **điều kiện số dương** của đề không phải chi tiết thừa.

In [25]:
assert max_product_diff_with_negatives([-10, -9, 1, 2, 3]) == 88 == max_product_diff_brute([-10, -9, 1, 2, 3])
print("Đáp án đúng khi có số âm:", 88)

Đáp án đúng khi có số âm: 88


## 15. Bài tập tự luyện (TODO)

### Bài 1 — Tính tay

Không chạy code: đáp án với `[1,2,3,4]`, `[9,9,9,9,1]`, `[2,7,1,8,2,8]`? Ghi vào `ex1_answers`.

In [26]:
ex1_answers = None   # TODO: list 3 số nguyên

### Bài 2 — Điền chỗ trống: cập nhật top-2

In [27]:
def ex2_top2(nums):
    big1 = big2 = float("-inf")
    for x in nums:
        # TODO: nếu x > big1 ...
        # TODO: elif x > big2 ...
        pass
    return big1, big2

# assert ex2_top2([4, 2, 5, 9, 7, 4, 8]) == (9, 8)
# assert ex2_top2([3, 3]) == (3, 3)

### Bài 3 — LeetCode 1464: Maximum Product of Two Elements

Trả về `(nums[i]-1)*(nums[j]-1)` lớn nhất với `i != j`. Yêu cầu `O(n)`.

In [28]:
def max_product_two(nums):
    # TODO
    pass

# assert max_product_two([3, 4, 5, 2]) == 12
# assert max_product_two([1, 5, 4, 5]) == 16

### Bài 4 — Tự viết lời giải một lượt (không nhìn code trên)

In [29]:
def ex4_max_product_difference(nums):
    # TODO
    pass

# for nums, e in examples: assert ex4_max_product_difference(nums) == e

### Bài 5 — LeetCode 628: tích lớn nhất của 3 số (có số âm)

Gợi ý: đáp án là max của `(3 số lớn nhất)` và `(2 số nhỏ nhất × số lớn nhất)`.

In [30]:
def maximum_product_three(nums):
    # TODO
    pass

# assert maximum_product_three([1, 2, 3]) == 6
# assert maximum_product_three([-10, -10, 1, 3, 2]) == 300
# assert maximum_product_three([-1, -2, -3]) == -6

### Bài 6 — Mini challenge: product difference với `k` cặp

Mọi số dương, `len(nums) >= 4k`. Chọn `4k` chỉ số phân biệt, chia thành `k` cặp "cộng" và `k` cặp "trừ"; tối đa hóa `Σ(tích cặp cộng) − Σ(tích cặp trừ)`. Câu hỏi then chốt: trong mỗi nhóm, nên **ghép** các số với nhau thế nào?

In [31]:
def max_k_pairs_difference(nums, k):
    # TODO
    pass

# assert max_k_pairs_difference([5, 6, 2, 7, 4], 1) == 34
# assert max_k_pairs_difference([1, 2, 3, 4, 5, 6, 7, 8], 2) == (8*7 + 6*5) - (1*4 + 2*3)

## 16. Gợi ý 3 mức

**Bài 1.** (1) Sắp xếp. (2) `[1,2,3,4]`: `4*3 - 1*2`. (3) `[9,9,9,9,1]`: hai nhỏ nhất là `1, 9`.

**Bài 2.** (1) Hai hạng. (2) Đẩy `big1` xuống trước khi ghi đè. (3) `big2, big1 = big1, x` / `elif x > big2: big2 = x`.

**Bài 3.** (1) Chỉ cần 2 số lớn nhất. (2) Dùng lại top-2. (3) `(big1 - 1) * (big2 - 1)`.

**Bài 4.** (1) 4 biến. (2) Khởi tạo nhỏ = `inf`. (3) Hai khối `if/elif` độc lập.

**Bài 5.** (1) Sắp xếp `s`. (2) Ứng viên `s[-1]*s[-2]*s[-3]` và `s[0]*s[1]*s[-1]`. (3) `max` của hai ứng viên.

**Bài 6.** (1) Phần cộng dùng `2k` số lớn nhất, phần trừ dùng `2k` số nhỏ nhất. (2) Bất đẳng thức hoán vị: tổng tích **lớn nhất** khi ghép lớn với lớn (liền kề sau khi sắp), tổng tích **nhỏ nhất** khi ghép lớn với nhỏ. (3) Sắp xếp `s`; `plus = Σ s[-1-2i]*s[-2-2i]`, `minus = Σ s[i]*s[2k-1-i]`, với `i` từ 0 tới `k-1`.

## 17. Đáp án tham khảo (chỉ xem sau khi tự làm)

In [32]:
ans_ex1 = [10, 72, 62]


def ans_ex2_top2(nums):
    big1 = big2 = float("-inf")
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        elif x > big2:
            big2 = x
    return big1, big2


def ans_max_product_two(nums):
    b1, b2 = ans_ex2_top2(nums)
    return (b1 - 1) * (b2 - 1)


def ans_ex4_max_product_difference(nums):
    big1 = big2 = 0
    small1 = small2 = float("inf")
    for x in nums:
        if x > big1:
            big2, big1 = big1, x
        elif x > big2:
            big2 = x
        if x < small1:
            small2, small1 = small1, x
        elif x < small2:
            small2 = x
    return big1 * big2 - small1 * small2


def ans_maximum_product_three(nums):
    s = sorted(nums)
    return max(s[-1] * s[-2] * s[-3], s[0] * s[1] * s[-1])


def ans_max_k_pairs_difference(nums, k):
    s = sorted(nums)
    plus = sum(s[-1 - 2 * i] * s[-2 - 2 * i] for i in range(k))
    minus = sum(s[i] * s[2 * k - 1 - i] for i in range(k))
    return plus - minus


assert ans_ex1 == [max_product_diff_sort(x) for x in [[1, 2, 3, 4], [9, 9, 9, 9, 1], [2, 7, 1, 8, 2, 8]]]
assert ans_ex2_top2([4, 2, 5, 9, 7, 4, 8]) == (9, 8) and ans_ex2_top2([3, 3]) == (3, 3)
assert ans_max_product_two([3, 4, 5, 2]) == 12 and ans_max_product_two([1, 5, 4, 5]) == 16
for nums, e in examples:
    assert ans_ex4_max_product_difference(nums) == e
assert ans_maximum_product_three([1, 2, 3]) == 6
assert ans_maximum_product_three([-10, -10, 1, 3, 2]) == 300
assert ans_maximum_product_three([-1, -2, -3]) == -6
assert ans_max_k_pairs_difference([5, 6, 2, 7, 4], 1) == 34
assert ans_max_k_pairs_difference([1, 2, 3, 4, 5, 6, 7, 8], 2) == (8 * 7 + 6 * 5) - (1 * 4 + 2 * 3)


# Brute force kiểm chứng bài 6 (k = 2): thử mọi tập 4 số cho phần cộng, 4 số cho phần trừ, mọi cách ghép
from itertools import combinations

def best_pairing(vals, want_max):
    a, b_, c_, d = vals
    sums = [a * b_ + c_ * d, a * c_ + b_ * d, a * d + b_ * c_]
    return max(sums) if want_max else min(sums)

def brute_k2(nums):
    idx = range(len(nums))
    best = float("-inf")
    for plus in combinations(idx, 4):
        rest = [i for i in idx if i not in plus]
        p = best_pairing([nums[i] for i in plus], True)
        for minus in combinations(rest, 4):
            best = max(best, p - best_pairing([nums[i] for i in minus], False))
    return best

random.seed(66)
for _ in range(60):
    arr = [random.randint(1, 9) for _ in range(random.randint(8, 10))]
    assert ans_max_k_pairs_difference(arr, 2) == brute_k2(arr), arr
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


Ở Bài 6, phần trừ ghép `1*4 + 2*3 = 10` chứ không phải `1*2 + 3*4 = 14`: muốn tổng tích **nhỏ nhất** thì ghép lớn với nhỏ (**bất đẳng thức hoán vị**). Brute force ở trên đã xác nhận công thức trên mảng ngẫu nhiên.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxProductDifference(self, nums: List[int]) -> int:
        big1 = big2 = 0
        small1 = small2 = float('inf')
        for x in nums:
            if x > big1:   big2, big1 = big1, x
            elif x > big2: big2 = x
            if x < small1:   small2, small1 = small1, x
            elif x < small2: small2 = x
        return big1 * big2 - small1 * small2
        # hoặc: nums.sort(); return nums[-1]*nums[-2] - nums[0]*nums[1]
```

**Checklist trước khi submit:**

- [ ] `from typing import List` nếu chạy ngoài LeetCode?
- [ ] Cập nhật top-2 bằng `if/elif`, đẩy hạng cũ xuống **trước**?
- [ ] Khởi tạo `small = inf`, `big = 0` (hoặc `-inf`)?
- [ ] Không dùng `set` (mất phần tử trùng)?
- [ ] Nhớ rằng insight dựa vào **mọi số dương**?